# Track 1 — Can It Play DOOM? (Google Colab A100 Setup)

This notebook sets up the repository on a **Google Colab A100** runtime using `uv` (Python 3.13 + `vizdoom==1.3.3` + `doom_env` + CUDA `torch`), verifies **Stage 1 (Connect)** for Training Role 1 (`submission/obs.py` + `submission/policy.py` + `submission/agent.py`), and provides the rollout/PPO integration template for Role 2.

In [ ]:
!nvidia-smi
!nproc

# Optional: mount Google Drive for persistent PPO checkpoints across Colab sessions
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/RohitBharadwaj-rvu/track-1.git"
BRANCH = "role1/telemetry-policy"  # switch to "master" after merge
REPO_DIR = Path("/content/track1")

if not REPO_DIR.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git fetch origin && git checkout {BRANCH} && git pull --ff-only

!bash training/colab_setup.sh

In [ ]:
%cd /content/track1

# 1. Export smoke weights, reload via DoomAgent on CUDA, and verify 2-map reset
!.venv/bin/python training/export_smoke_weights.py --device cuda --out /tmp/smoke_policy.pt

# 2. Quick A100 sequence chunk throughput benchmark (T=64, B=32)
!.venv/bin/python - <<'PY'
import time
import torch
from submission.policy import PolicyConfig, build_policy

device = "cuda" if torch.cuda.is_available() else "cpu"
policy = build_policy(PolicyConfig(arch="telemetry_gru_v1")).to(device)
T, B = 64, 32
obs_seq = {
    "ray_dist": torch.rand(T, B, 64, device=device),
    "ray_type": torch.randint(0, 6, (T, B, 64), device=device),
    "ent_feat": torch.randn(T, B, 32, 5, device=device),
    "ent_type": torch.randint(0, 12, (T, B, 32), device=device),
    "ent_mask": torch.randint(0, 2, (T, B, 32), device=device, dtype=torch.float32),
    "vitals": torch.randn(T, B, 17, device=device),
}
prev_actions = torch.randint(0, 2, (T, B, 9), device=device, dtype=torch.float32)
actions = torch.randint(0, 2, (T, B, 9), device=device, dtype=torch.float32)
starts = torch.zeros(T, B, device=device, dtype=torch.bool)
starts[17, 3] = True
init_hidden = policy.init_hidden(B, device)

for _ in range(5):
    log_prob, entropy, value = policy.evaluate_chunks(obs_seq, prev_actions, actions, init_hidden, starts)
    (-(log_prob.mean() + 0.01 * entropy.mean()) + 0.5 * (value**2).mean()).backward()
    policy.zero_grad(set_to_none=True)
if device == "cuda":
    torch.cuda.synchronize()

t0 = time.perf_counter()
for _ in range(50):
    log_prob, entropy, value = policy.evaluate_chunks(obs_seq, prev_actions, actions, init_hidden, starts)
    (-(log_prob.mean() + 0.01 * entropy.mean()) + 0.5 * (value**2).mean()).backward()
    policy.zero_grad(set_to_none=True)
if device == "cuda":
    torch.cuda.synchronize()
dt = time.perf_counter() - t0
print(f"A100/GPU fwd+bwd: {50 * T * B / dt:,.0f} steps/s ({dt / 50 * 1000:.2f} ms per {T}x{B} minibatch)")
PY

In [ ]:
%cd /content/track1
!.venv/bin/pytest -q
!.venv/bin/python validate.py --device cuda --map maps/two_rooms.wad --map maps/two_rooms.wad

In [ ]:
# Template for Role 2 (PPO + Rollout Collection) on A100:
# Uses the frozen submission/obs.py + submission/policy.py contract.

from pathlib import Path
import torch
from submission.obs import batch_preprocessed_obs, obs_to_torch, preprocess_obs
from submission.policy import PolicyConfig, build_policy, load_checkpoint, save_checkpoint

CKPT_DIR = Path("/content/drive/MyDrive/track1_checkpoints") if Path("/content/drive/MyDrive").exists() else Path("/tmp/track1_checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = PolicyConfig(arch="telemetry_gru_v1")
policy = build_policy(cfg).to(device)
optimizer = torch.optim.AdamW(policy.parameters(), lr=3e-4, eps=1e-5)

print("Policy params:", sum(p.numel() for p in policy.parameters()), "| checkpoint dir:", CKPT_DIR)
# During rollout step:
# obs_t = obs_to_torch(batch_preprocessed_obs([preprocess_obs(o) for o in obs_list]), device=device)
# logits, value, next_hidden = policy.act_step(obs_t, prev_action, hidden)
# (When done[b] resets env b before step t, zero both hidden[:, b] and prev_action[b] and set episode_starts[t, b] = True)
#
# Export checkpoint for DoomAgent / validate.py:
# save_checkpoint("submission/weights/policy.pt", policy, cfg, meta={"step": global_step})